# Black Swan Logic v8 — Quickstart

สมุดนี้รัน Logic Anomaly Detection จาก package `black_swan` ใน repository นี้โดยตรง และแสดง Point / Temporal / Collective evidence พร้อม final route.

> Repository เป็น private: เมื่อเปิดผ่านปุ่ม **Open in Colab** ให้ authorize GitHub กับ Colab ก่อน หาก environment ยังไม่มี source ให้รันเซลล์ setup ด้านล่างหลัง clone repository ใน Colab.

In [ ]:
from pathlib import Path
import sys

candidates = [Path('/content/back_swan'), Path.cwd(), Path.cwd().parent]
root = next((p for p in candidates if (p / 'src' / 'black_swan').exists()), None)
if root is None:
    raise RuntimeError(
        'back_swan source not found. Because this repository is private, authorize GitHub in Colab, '
        'clone narongdetpyou-ux/back_swan into /content/back_swan, then rerun this cell.'
    )
sys.path.insert(0, str(root / 'src'))
print('Using repository:', root)


In [ ]:
from black_swan import BlackSwanV8, Calibration, ScenarioInput

calibration = Calibration(
    composite_threshold=1.5,
    quantile=0.99,
    calibration_size=500,
    target_false_alert_rate=0.01,
)
engine = BlackSwanV8(calibration)

# Stable 96-step baseline with a small daily pattern.
history = [100 + ((i % 24) - 12) * 0.05 for i in range(96)]

normal = ScenarioInput(
    scenario_id='quickstart-normal',
    history={'metric_a': history},
    recent={'metric_a': [100.0, 100.1, 99.9, 100.0]},
)

anomaly = ScenarioInput(
    scenario_id='quickstart-anomaly',
    history={'metric_a': history},
    recent={'metric_a': [108.0, 112.0, 118.0, 125.0]},
)

for case in (normal, anomaly):
    decision = engine.decide(case)
    print('\n', case.scenario_id)
    print('route       :', decision.final_route)
    print('point       :', round(decision.point_score, 3))
    print('temporal    :', round(decision.temporal_score, 3))
    print('collective  :', round(decision.collective_score, 3))
    print('composite   :', round(decision.composite_score, 3))
    print('threshold   :', decision.calibrated_threshold)
    print('uncertainty :', round(decision.uncertainty, 3))
    print('reason      :', decision.reason_code)


## สิ่งที่ควรสังเกต

- `point_score` จับการเบี่ยงเบนเด่นของจุดข้อมูล
- `temporal_score` สะสมความผิดปกติตามลำดับเวลา
- `collective_score` รวมสัญญาณจากหลาย metric เมื่อใช้ข้อมูลหลายช่องทาง
- `composite_score` เทียบกับ calibrated threshold ก่อนส่งต่อไปยัง critic/context และ final route
- ผลใน quickstart เป็น synthetic demonstration ไม่ใช่ production calibration